# Chapter 14 — Q-Learning

Learning objectives:
- Implement Q-learning: Q(s,a) ← Q(s,a) + α[r + γ max Q(s',a') − Q(s,a)]
- Apply to 5×5 gridworld
- Compare greedy policy and value function after training

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
print("Ready")

## Environment: 5×5 Gridworld

Start (0,0), goal (4,4). Reward: +10 at goal, -1 per step. Wall: stay + -1.

In [ ]:
def gridworld_step(state, action):
    """4 actions: 0=up,1=down,2=left,3=right. Returns next_state, reward, done."""
    row, col = state
    dr = [-1, 1, 0, 0]
    dc = [0, 0, -1, 1]
    nr = row + dr[action]
    nc = col + dc[action]
    if not (0 <= nr < 5 and 0 <= nc < 5):
        return state, -1, False
    if (nr, nc) == (4, 4):
        return (4, 4), 10, True
    return (nr, nc), -1, False

## Exercise: Q-learning training loop

In [ ]:
def q_learning(n_episodes=2000, alpha=0.1, gamma=0.99, epsilon=0.1):
    """Train Q-learning on the 5x5 gridworld.
    Returns Q: dict mapping (state, action) -> float.
    """
    n_actions = 4
    Q = {}   # Q[(state, action)] = value

    for ep in range(n_episodes):
        state = (0, 0)
        done = False
        steps = 0
        while not done and steps < 50:
            # TODO: epsilon-greedy action selection
            # action = ...

            # TODO: take step
            # next_state, reward, done = gridworld_step(state, action)

            # TODO: Q-learning update:
            # target = reward + gamma * max_a Q[(next_state, a)] (if not done, else reward)
            # Q[(state, action)] += alpha * (target - Q[(state, action)])

            state = None  # update state
            steps += 1
    return Q

Q = q_learning()
print(f"Q-table size: {len(Q)} entries")

In [ ]:
def q_learning(n_episodes=2000, alpha=0.1, gamma=0.99, epsilon=0.1):
    n_actions = 4
    Q = {}

    def get_q(s, a):
        return Q.get((s, a), 0.0)

    def best_action(s):
        qs = [get_q(s, a) for a in range(n_actions)]
        return int(np.argmax(qs))

    for ep in range(n_episodes):
        state = (0, 0)
        done = False
        steps = 0
        while not done and steps < 50:
            if np.random.random() < epsilon:
                action = np.random.randint(n_actions)
            else:
                action = best_action(state)
            next_state, reward, done = gridworld_step(state, action)
            if done:
                target = reward
            else:
                target = reward + gamma * max(get_q(next_state, a) for a in range(n_actions))
            Q[(state, action)] = get_q(state, action) + alpha * (target - get_q(state, action))
            state = next_state
            steps += 1
    return Q

Q = q_learning()

# Plot value function
V = np.zeros((5, 5))
for r in range(5):
    for c in range(5):
        V[r, c] = max(Q.get(((r,c), a), 0.0) for a in range(4))

plt.figure(figsize=(5, 4))
plt.imshow(V, cmap='Blues')
plt.colorbar(label='max_a Q(s,a)')
plt.title('Value Function after Q-Learning')
for r in range(5):
    for c in range(5):
        plt.text(c, r, f'{V[r,c]:.1f}', ha='center', va='center', fontsize=7)
plt.tight_layout()
plt.show()